In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


import nvitk as nv
from nvitk import db

In [2]:
repo, xnat_config = db.get_repo_from_settings(return_xnat_config=True)

Using local root: ~/nvitk/dataset/nvitk-dataset


In [ ]:
aux = repo.image(modality='4dflow', variables=['flow_mean'], wide=False)
aux = aux[(aux['variable_id' ] == 'flow_mean') & (aux['value_num'].notna())]

_SUBJECTS = pd.unique(aux.subject_uid)
_CLINICAL_VARS = ['age_at_mri', 'apoe_group', 'apoe', 'bmi', 'pp', 'map', 'hematocrit', 'pedframi10', 'pedframi30', 'sex', 'score2', 'tacsctot_group']
_PLAQUE_VARS = ['total_carotid_plaque_vol', 'total_femoral_plaque_vol', 'total_plaque_vol', 'right_carotid_plaque_vol', 'left_carotid_plaque_vol']
_COGNITIVE_VARS = ['z_compo_processingspeed', 'z_compo_globalfull', 'z_compo_epismemory', 'z_compo_attworkmemospeed']
_FLOW_VARS = ['flow_mean', 'pi']
_ASL_VARS = ['mean_cbf', 'att_mean']

TERRITORY_FLOW_REGIONS: dict[str, tuple[str, ...]] = {
    "ICA": ("lica", "rica"),
    "Venous": ("sssv", "strv", "ltsv", "rtsv"),
    "ACA": ( "laca", "raca"),
    "MCA": ("lmca", "rmca"),
    "PCA": ("lpca", "rpca"),
    "Basilar": ("basi",),
}

TERRITORY_ASL_V8_REGIONS: dict[str, tuple[str, ...]] = {
    "ACA": (
        "left_aca_8",
        "right_aca_8",
    ),
    "MCA": (
        "left_mca_8",
        "right_mca_8",
    ),
    "PCA": (
        "left_pca_8",
        "right_pca_8",
    ),
    "Basilar": (
        "left_basilar_8",
        "right_basilar_8",
    ),
    "Watershed": ("watershed_0", "watershed_8"),
}

In [13]:
clinical  = repo.clinical(variables=_CLINICAL_VARS, filters={'subject_uid': {'$in': _SUBJECTS}})
plaque    = repo.clinical(variables=_PLAQUE_VARS, filters={'subject_uid': {'$in': _SUBJECTS}})
cognitive = repo.cognitive(variables=_COGNITIVE_VARS, filters={'subject_uid': {'$in': _SUBJECTS}})
clinical   = repo.join([clinical, plaque, cognitive])

flow      = repo.image(modality='4dflow', variables=_FLOW_VARS, filters={'subject_uid': {'$in': _SUBJECTS}})
asl       = repo.image(modality='asl', variables=_ASL_VARS, atlas='vascular-8', filters={'subject_uid': {'$in': _SUBJECTS}})
image_df = repo.join([flow, asl])

territory_df = nv.stats.melt_imaging_territories(
    image_df, id_cols=['subject_uid'], 
    flow_vars=_FLOW_VARS, asl_vars=_ASL_VARS, 
    territory_flow_regions=TERRITORY_FLOW_REGIONS,
    territory_asl_v8_regions=TERRITORY_ASL_V8_REGIONS,
    include_frame_index=False
)

In [14]:
territory_df

,subject_uid,territory,modality_group,region_id,variable_id,value
0,PESA1006009,Posterior Circulation,flow,basilar,flow_mean,248.35062
1,PESA10061584,Posterior Circulation,flow,basilar,flow_mean,195.941298
2,PESA10067929,Posterior Circulation,flow,basilar,flow_mean,244.387905
3,PESA10086976,Posterior Circulation,flow,basilar,flow_mean,379.135876
4,PESA10112400,Posterior Circulation,flow,basilar,flow_mean,206.517756
...,...,...,...,...,...,...
18095,PESA9897316,WWW,asl,watershed_8,mean_cbf,42.002036
18096,PESA9928801,WWW,asl,watershed_8,mean_cbf,49.882072
18097,PESA9935104,WWW,asl,watershed_8,mean_cbf,46.629408
18098,PESA9947716,WWW,asl,watershed_8,mean_cbf,55.293241


CACS

In [5]:
_IMAGING_VARS = ['mean_cbf']
_COVAR_FOR_MODEL = [
    "tacsctot_group",
    "age_at_mri",
    "sex",
    "hematocrit",
    # "score2",
]
_cov = [c for c in _COVAR_FOR_MODEL if c in clinical.columns]

analysis_df = nv.stats.build_analysis_df_from_repo_frames(
    territory_df,
    clinical,
    imaging_variable_ids=_IMAGING_VARS,
    covariate_cols=_cov,
)

analysis_df = analysis_df.dropna().reset_index(drop=True)
analysis_df["age_c"] = analysis_df["age_at_mri"] - float(analysis_df["age_at_mri"].mean())
analysis_df = analysis_df.drop(columns=['age_at_mri'])
analysis_df['territory'] = pd.Categorical(analysis_df['territory'], categories=list(TERRITORY_ASL_V8_REGIONS.keys()), ordered=True)
analysis_df['tacsctot_group'] = pd.Categorical(analysis_df['tacsctot_group'], categories=['g0', 'g1', 'g2', 'g3'], ordered=True)
analysis_df

,subject_uid,territory,mean_cbf,tacsctot_group,sex,hematocrit,age_c
0,PESA1006009,ACA,58.483761,g0,1,47.0,5.095198
1,PESA10061584,ACA,40.072780,g2,1,48.0,6.495198
2,PESA10067929,ACA,59.831661,g0,1,42.0,6.765198
3,PESA10086976,ACA,57.035611,g0,0,45.0,-4.584802
4,PESA10112400,ACA,56.868346,g0,1,48.0,-4.034802
...,...,...,...,...,...,...,...
1765,PESA9897316,Watershed,42.002036,g1,1,52.0,-2.424802
1766,PESA9928801,Watershed,49.882072,g0,0,45.0,4.355198
1767,PESA9935104,Watershed,46.629408,g0,0,45.0,2.815198
1768,PESA9947716,Watershed,55.293241,g1,1,42.0,1.355198


In [6]:
formula = (
    f"mean_cbf ~ C(tacsctot_group, Treatment('g0')) "
    f"* C(territory, Treatment('MCA')) "
    f"+ age_c + sex + hematocrit"
)

_model_cols = [
    "mean_cbf",
    "tacsctot_group",
    "territory",
    "age_c",
    "sex",
    "hematocrit",
    "subject_uid",
]

res, df_fit, _meta = nv.stats.fit_or_load_mixedlm(
    model_path=None,
    data=analysis_df,
    formula=formula,
    groups="territory",
    re_formula="0",
    vc_formula={"subject": "0 + C(subject_uid)"},
    overwrite=False,
    required_columns=_model_cols,
)

nv.stats.print_mixedlm_info(
    res,
    outcome_name="mean_cbf",
    group_name="territory",
    vc_group_name="subject_uid",
    output_path=None,
)


LinAlgError: Singular matrix

In [ ]:
fig = nv.stats.plot_mixedlm_params(
    result=res,
    df_fit=df_fit,
    x="tacsctot_group",
    y="mean_cbf",
    group="territory",
    mode="auto",
    include_points=True,
    output_path=None,
    title="mean_cbf mixed model",
    x_label="tacsctot_group",
    y_label="mean_cbf",
    covariate_refs={"sex": 0.0, "hematocrit": float(df_fit["hematocrit"].mean()), "score2": float(df_fit["score2"].mean())},
)
fig.clf()

/home/imarcoss/nvitk/src/nvitk/stats/mixedlm.py:346: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.pointplot(


<Figure size 1000x600 with 0 Axes>

pi age

In [ ]:
_IMAGING_VARS = ['pi']
_COVAR_FOR_MODEL = [
    "tacsctot_group",
    "age_at_mri",
    "sex",
    "hematocrit",
    # "score2",
]
_cov = [c for c in _COVAR_FOR_MODEL if c in clinical.columns]

ONE2ONE_FLOW_REGIONS = {
    "RICA": ("rica"),
    "LCA": ("lica"),
    "SSSV": ("sssv"),
    "STRV": ("strv"),
    "LTSV": ("ltsv"),
    "RTSV": ("rtsv"),
    "LACA": ("laca"),
    "RACA": ("raca"),
    "LMCA": ("lmca"),
    "RMCA": ("rmca"),
    "LPCA": ("lpca"),
    "RPCA": ("rpca"),
    "Basi": ("basi"),
}
territory_df = nv.stats.melt_imaging_territories(
    image_df, id_cols=['subject_uid'], 
    flow_vars=_IMAGING_VARS, 
    territory_flow_regions=ONE2ONE_FLOW_REGIONS,
    include_frame_index=False
)

analysis_df = nv.stats.build_analysis_df_from_repo_frames(
    territory_df,
    clinical,
    imaging_variable_ids=_IMAGING_VARS,
    covariate_cols=_cov,
)

analysis_df = analysis_df.dropna().reset_index(drop=True)
analysis_df = analysis_df.dropna().reset_index(drop=True)
analysis_df["age_c"] = analysis_df["age_at_mri"] - float(analysis_df["age_at_mri"].mean())
analysis_df = analysis_df.drop(columns=['age_at_mri'])
# analysis_df['territory'] = pd.Categorical(analysis_df['territory'], categories=list(ONE2ONE_FLOW_REGIONS.keys()), ordered=True)
analysis_df = analysis_df[analysis_df['territory'] != 'Unmapped']
analysis_df['tacsctot_group'] = pd.Categorical(analysis_df['tacsctot_group'], categories=['g0', 'g1', 'g2', 'g3'], ordered=True)
territory_df

,subject_uid,territory,modality_group,region_id,variable_id,value
0,PESA1006009,Posterior Circulation,flow,basilar,pi,0.789627
1,PESA10061584,Posterior Circulation,flow,basilar,pi,0.806241
2,PESA10067929,Posterior Circulation,flow,basilar,pi,0.658304
3,PESA10086976,Posterior Circulation,flow,basilar,pi,0.698962
4,PESA10112400,Posterior Circulation,flow,basilar,pi,0.688548
...,...,...,...,...,...,...
12303,PESA9897316,Watershed,asl,watershed_8,mean_cbf,42.002036
12304,PESA9928801,Watershed,asl,watershed_8,mean_cbf,49.882072
12305,PESA9935104,Watershed,asl,watershed_8,mean_cbf,46.629408
12306,PESA9947716,Watershed,asl,watershed_8,mean_cbf,55.293241


In [ ]:
formula = (
    f"pi ~ age_c + sex"
)

_model_cols = [
    "pi",
    "territory",
    "age_c",
    "sex",
    "subject_uid",
]

res, df_fit, _meta = nv.stats.fit_or_load_mixedlm(
    model_path=None,
    data=analysis_df,
    formula=formula,
    groups="territory",
    re_formula="1 + age_c",
    vc_formula={"subject": "0 + C(subject_uid)"},
    overwrite=False,
    required_columns=_model_cols,
)

nv.stats.print_mixedlm_info(
    res,
    outcome_name="pi",
    group_name="territory",
    vc_group_name="subject_uid",
    output_path=None,
)


MemoryError: Unable to allocate 18.9 GiB for an array with shape (50404, 50404) and data type float64